# Investment Diligence ETL Pipeline

Pulls financial statement data from **SEC EDGAR** and **Alpha Vantage**, reconciles the two sources against each other, and exports CSVs for MySQL.

Run every cell top to bottom. You'll be prompted for an Alpha Vantage API key and a SEC EDGAR User-Agent string.

No installs required beyond what Colab ships with (`requests`, `pandas` are preinstalled).

In [ ]:
import requests
import pandas as pd
import time
import re
from datetime import datetime
from getpass import getpass

pd.set_option('display.max_columns', None)

## 1. Configuration

SEC EDGAR requires every request to send a descriptive `User-Agent` header (name + email) — it's a courtesy policy so they can contact you if a script misbehaves, not a real registration.

Alpha Vantage's free key is instant: https://www.alphavantage.co/support/#api-key

In [ ]:
SEC_USER_AGENT = input("SEC EDGAR User-Agent (e.g. 'Jane Doe jane@example.com'): ").strip()
ALPHA_VANTAGE_KEY = getpass("Alpha Vantage API key: ").strip()

SEC_HEADERS = {"User-Agent": SEC_USER_AGENT}

# Tickers to analyze -- edit this list freely
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "JPM", "KO", "XOM"]

# How many fiscal years back to pull
YEARS_BACK = 5

# Discrepancy threshold (%) above which we flag a metric as needing review
DISCREPANCY_THRESHOLD_PCT = 5.0

print(f"Configured for {len(TICKERS)} tickers: {TICKERS}")

## 2. SEC EDGAR: ticker -> CIK lookup

In [ ]:
def get_ticker_to_cik_map():
    """SEC publishes a single JSON file mapping every ticker to its CIK."""
    url = "https://www.sec.gov/files/company_tickers.json"
    resp = requests.get(url, headers=SEC_HEADERS)
    resp.raise_for_status()
    raw = resp.json()
    mapping = {}
    for entry in raw.values():
        mapping[entry["ticker"].upper()] = str(entry["cik_str"]).zfill(10)
    return mapping

ticker_to_cik = get_ticker_to_cik_map()
missing = [t for t in TICKERS if t not in ticker_to_cik]
if missing:
    print(f"WARNING: no CIK found for {missing}, they will be skipped")

cik_map = {t: ticker_to_cik[t] for t in TICKERS if t in ticker_to_cik}
cik_map

## 3. SEC EDGAR: pull annual financial facts

We pull from the `companyfacts` endpoint and extract the US-GAAP tags we care about, keeping only 10-K (annual, `FY`) values.

In [ ]:
# GAAP tag -> our normalized metric name. Some companies use alternate tags
# for the same concept, so we list fallbacks and take the first that exists.
GAAP_TAGS = {
    "revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "net_income": ["NetIncomeLoss"],
    "total_assets": ["Assets"],
    "total_liabilities": ["Liabilities"],
    "total_equity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities"],
    "eps_diluted": ["EarningsPerShareDiluted"],
}

def get_sec_company_facts(cik):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    resp = requests.get(url, headers=SEC_HEADERS)
    if resp.status_code != 200:
        return None
    return resp.json()

def extract_annual_metric(facts_json, tag_candidates):
    """Return {fiscal_year: value} for the first matching GAAP tag, FY/10-K only."""
    us_gaap = facts_json.get("facts", {}).get("us-gaap", {})
    for tag in tag_candidates:
        if tag not in us_gaap:
            continue
        units = us_gaap[tag].get("units", {})
        values_by_year = {}
        for unit_key, entries in units.items():
            for e in entries:
                if e.get("fp") == "FY" and e.get("form") == "10-K" and e.get("fy"):
                    # Prefer the most recently filed value for a given fiscal year
                    values_by_year[e["fy"]] = e["val"]
        if values_by_year:
            return values_by_year
    return {}

sec_records = []  # one row per (ticker, fiscal_year)
current_year = datetime.now().year

for ticker, cik in cik_map.items():
    print(f"Pulling SEC data for {ticker} (CIK {cik})...")
    facts = get_sec_company_facts(cik)
    time.sleep(0.15)  # be polite to SEC's servers
    if facts is None:
        print(f"  -> no data returned, skipping")
        continue

    entity_name = facts.get("entityName", ticker)
    metrics_by_year = {}
    for metric_name, tag_candidates in GAAP_TAGS.items():
        yearly = extract_annual_metric(facts, tag_candidates)
        for fy, val in yearly.items():
            if fy < current_year - YEARS_BACK:
                continue
            metrics_by_year.setdefault(fy, {})[metric_name] = val

    for fy, metrics in metrics_by_year.items():
        row = {"ticker": ticker, "company_name": entity_name, "cik": cik,
               "fiscal_year": fy, "source": "SEC_EDGAR"}
        row.update(metrics)
        sec_records.append(row)

sec_df = pd.DataFrame(sec_records)
print(f"\nCollected {len(sec_df)} SEC EDGAR annual records")
sec_df.head()

## 4. Alpha Vantage: pull annual fundamentals

Free tier = 5 calls/minute, 25/day. We use `OVERVIEW`, `INCOME_STATEMENT`, `BALANCE_SHEET`, and `CASH_FLOW`, pacing requests so we never get throttled.

In [ ]:
AV_BASE_URL = "https://www.alphavantage.co/query"
AV_CALLS_MADE = 0

def av_request(params):
    """Wraps requests to Alpha Vantage with basic rate-limit pacing and error checks."""
    global AV_CALLS_MADE
    params = {**params, "apikey": ALPHA_VANTAGE_KEY}
    resp = requests.get(AV_BASE_URL, params=params)
    resp.raise_for_status()
    data = resp.json()
    AV_CALLS_MADE += 1
    if "Note" in data or "Information" in data:
        # Rate limit hit -- back off and retry once
        print("  Rate limit message from Alpha Vantage, waiting 60s...")
        time.sleep(60)
        resp = requests.get(AV_BASE_URL, params=params)
        data = resp.json()
    if AV_CALLS_MADE % 5 == 0:
        time.sleep(65)  # stay under the 5-calls-per-minute ceiling
    else:
        time.sleep(1)
    return data

def safe_float(val):
    try:
        if val in (None, "None", "-"):
            return None
        return float(val)
    except (ValueError, TypeError):
        return None

av_records = []

for ticker in TICKERS:
    print(f"Pulling Alpha Vantage data for {ticker}...")
    income = av_request({"function": "INCOME_STATEMENT", "symbol": ticker})
    balance = av_request({"function": "BALANCE_SHEET", "symbol": ticker})
    cashflow = av_request({"function": "CASH_FLOW", "symbol": ticker})

    income_reports = {r["fiscalDateEnding"]: r for r in income.get("annualReports", [])}
    balance_reports = {r["fiscalDateEnding"]: r for r in balance.get("annualReports", [])}
    cashflow_reports = {r["fiscalDateEnding"]: r for r in cashflow.get("annualReports", [])}

    all_dates = set(income_reports) | set(balance_reports) | set(cashflow_reports)

    for date_str in all_dates:
        fy = int(date_str[:4])
        if fy < datetime.now().year - YEARS_BACK:
            continue
        inc = income_reports.get(date_str, {})
        bal = balance_reports.get(date_str, {})
        cf = cashflow_reports.get(date_str, {})

        row = {
            "ticker": ticker,
            "fiscal_year": fy,
            "source": "ALPHA_VANTAGE",
            "revenue": safe_float(inc.get("totalRevenue")),
            "net_income": safe_float(inc.get("netIncome")),
            "total_assets": safe_float(bal.get("totalAssets")),
            "total_liabilities": safe_float(bal.get("totalLiabilities")),
            "total_equity": safe_float(bal.get("totalShareholderEquity")),
            "operating_cash_flow": safe_float(cf.get("operatingCashflow")),
        }
        av_records.append(row)

av_df = pd.DataFrame(av_records)
print(f"\nCollected {len(av_df)} Alpha Vantage annual records")
av_df.head()

## 5. Reconciliation: compare SEC EDGAR vs Alpha Vantage

For every (ticker, fiscal_year, metric) present in both sources, compute the % discrepancy. Anything above `DISCREPANCY_THRESHOLD_PCT` gets logged for review.

In [ ]:
METRICS_TO_RECONCILE = ["revenue", "net_income", "total_assets", "total_liabilities",
                         "total_equity", "operating_cash_flow"]

reconciliation_records = []
merged_records = []  # the reconciled, best-available value per metric

sec_indexed = sec_df.set_index(["ticker", "fiscal_year"]) if not sec_df.empty else pd.DataFrame()
av_indexed = av_df.set_index(["ticker", "fiscal_year"]) if not av_df.empty else pd.DataFrame()

all_keys = set(sec_indexed.index if not sec_df.empty else []) | set(av_indexed.index if not av_df.empty else [])

for ticker, fy in sorted(all_keys):
    sec_row = sec_indexed.loc[(ticker, fy)] if (ticker, fy) in sec_indexed.index else None
    av_row = av_indexed.loc[(ticker, fy)] if (ticker, fy) in av_indexed.index else None

    merged_row = {"ticker": ticker, "fiscal_year": fy}

    for metric in METRICS_TO_RECONCILE:
        sec_val = sec_row[metric] if sec_row is not None and metric in sec_row and pd.notna(sec_row[metric]) else None
        av_val = av_row[metric] if av_row is not None and metric in av_row and pd.notna(av_row[metric]) else None

        if sec_val is not None and av_val is not None and sec_val != 0:
            discrepancy_pct = abs(sec_val - av_val) / abs(sec_val) * 100
            if discrepancy_pct > DISCREPANCY_THRESHOLD_PCT:
                reconciliation_records.append({
                    "ticker": ticker, "fiscal_year": fy, "metric_name": metric,
                    "source_a": "SEC_EDGAR", "value_a": sec_val,
                    "source_b": "ALPHA_VANTAGE", "value_b": av_val,
                    "discrepancy_pct": round(discrepancy_pct, 2),
                })
            # SEC EDGAR is the primary/authoritative source when both agree closely
            merged_row[metric] = sec_val
        else:
            merged_row[metric] = sec_val if sec_val is not None else av_val

    # eps only comes from SEC in this pipeline
    merged_row["eps_diluted"] = sec_row["eps_diluted"] if sec_row is not None and "eps_diluted" in sec_row and pd.notna(sec_row["eps_diluted"]) else None

    merged_records.append(merged_row)

financial_statements_df = pd.DataFrame(merged_records)
reconciliation_log_df = pd.DataFrame(reconciliation_records)

print(f"Merged financial statements: {len(financial_statements_df)} rows")
print(f"Discrepancies flagged (> {DISCREPANCY_THRESHOLD_PCT}%): {len(reconciliation_log_df)} rows")
reconciliation_log_df.head(10)

## 6. Alpha Vantage: daily stock prices

Pulled separately since it's a different endpoint (`TIME_SERIES_DAILY`), trimmed to the last ~2 years to keep the table a reasonable size.

In [ ]:
price_records = []
PRICE_LOOKBACK_DAYS = 730

for ticker in TICKERS:
    print(f"Pulling price history for {ticker}...")
    data = av_request({"function": "TIME_SERIES_DAILY", "symbol": ticker, "outputsize": "full"})
    series = data.get("Time Series (Daily)", {})
    cutoff = datetime.now().timestamp() - PRICE_LOOKBACK_DAYS * 86400
    for date_str, values in series.items():
        d = datetime.strptime(date_str, "%Y-%m-%d")
        if d.timestamp() < cutoff:
            continue
        price_records.append({
            "ticker": ticker,
            "price_date": date_str,
            "open": safe_float(values.get("1. open")),
            "high": safe_float(values.get("2. high")),
            "low": safe_float(values.get("3. low")),
            "close": safe_float(values.get("4. close")),
            "volume": safe_float(values.get("5. volume")),
            "source": "ALPHA_VANTAGE",
        })

stock_prices_df = pd.DataFrame(price_records)
print(f"\nCollected {len(stock_prices_df)} daily price rows")
stock_prices_df.head()

## 7. Build the `companies` table and validate before export

Basic sanity checks: no negative revenue/assets, and a loose balance-sheet-equation check (`assets \u2248 liabilities + equity`). Rows that fail get printed so you can eyeball them -- they're still exported since MySQL's own CHECK constraints (step 2) are the real gate.


In [ ]:
companies_df = sec_df[["ticker", "company_name", "cik"]].drop_duplicates(subset=["ticker"]).reset_index(drop=True)
companies_df["company_id"] = companies_df.index + 1
companies_df = companies_df[["company_id", "ticker", "company_name", "cik"]]

ticker_to_id = dict(zip(companies_df["ticker"], companies_df["company_id"]))
financial_statements_df["company_id"] = financial_statements_df["ticker"].map(ticker_to_id)
stock_prices_df["company_id"] = stock_prices_df["ticker"].map(ticker_to_id)
reconciliation_log_df["company_id"] = reconciliation_log_df["ticker"].map(ticker_to_id) if not reconciliation_log_df.empty else None

# --- validation checks, printed for visibility ---
neg_revenue = financial_statements_df[financial_statements_df["revenue"] < 0]
if not neg_revenue.empty:
    print(f"WARNING: {len(neg_revenue)} rows with negative revenue")

def balance_check(row):
    a, l, e = row.get("total_assets"), row.get("total_liabilities"), row.get("total_equity")
    if pd.isna(a) or pd.isna(l) or pd.isna(e) or a == 0:
        return None
    return abs(a - (l + e)) / abs(a) * 100

financial_statements_df["_balance_check_pct"] = financial_statements_df.apply(balance_check, axis=1)
bad_balance = financial_statements_df[financial_statements_df["_balance_check_pct"] > 2.0]
if not bad_balance.empty:
    print(f"WARNING: {len(bad_balance)} rows where Assets != Liabilities + Equity (>2% off)")
    print(bad_balance[["ticker", "fiscal_year", "_balance_check_pct"]])

financial_statements_df = financial_statements_df.drop(columns=["_balance_check_pct"])

# Reorder / finalize columns for export
financial_statements_df = financial_statements_df[[
    "company_id", "ticker", "fiscal_year", "revenue", "net_income", "total_assets",
    "total_liabilities", "total_equity", "operating_cash_flow", "eps_diluted"
]]

print("\nValidation complete.")

## 8. Export CSVs and download

In [ ]:
companies_df.to_csv("companies.csv", index=False)
financial_statements_df.to_csv("financial_statements.csv", index=False)
stock_prices_df.to_csv("stock_prices.csv", index=False)
reconciliation_log_df.to_csv("reconciliation_log.csv", index=False)

print("Exported: companies.csv, financial_statements.csv, stock_prices.csv, reconciliation_log.csv")

try:
    from google.colab import files
    for f in ["companies.csv", "financial_statements.csv", "stock_prices.csv", "reconciliation_log.csv"]:
        files.download(f)
except ImportError:
    print("Not running in Colab -- files are saved in the current working directory.")

## Next step

Move these 4 CSVs into the project's `data/` folder, then follow the main `README.md`:
run `sql/01_schema.sql`, import the CSVs via MySQL Workbench's Table Data Import Wizard,
then run `sql/03_anomaly_detection.sql`.